# Classifier performance: final models

The four mmBERT classifiers (topic and format, each trained with and without the URL in the
input) after one epoch over the 6.85M training documents: 428,174 steps of 16 documents,
learning rate rising to 3.05e-5 over the first 7.6% of the steps and then decaying linearly
to 0. They are evaluated on the **test split** against the Qwen3.8-27B label distributions,
**without the test documents that duplicate training documents** (`EXCLUDE_LEAKED`;
section 11 shows the effect of leaving them in).

Each run's final model is the checkpoint with the lowest validation Jensen-Shannon
divergence among its 20 evaluations (`--load_best_model_at_end`, `--metric_for_best_model
js_divergence_bits`), which need not be the last one; the end of section 1 shows which
checkpoint each model is. The selection uses the validation split only, so the test split
stays unseen.

The half-epoch checkpoints on the validation split are analysed in
`classifier_half_epoch.ipynb`, from which this notebook was copied; the configuration cell
(`SPLIT`, `STEP`, `EXCLUDE_LEAKED`) switches between the two.

**Data.** The test split of `results/datasets/<label type>/hplt-finetune`: 364,402
documents, 3,000 per language (fewer for the 17 smaller languages); 362,706 after leaving
out the 1,696 that are near or exact duplicates of training documents. The no-URL datasets
hold the same documents and labels without the `url` column (checked below).

**Where the predictions come from.** At the end of training, `finetune.py` loads the best
checkpoint, saves it in the run directory (`results/models/<label type>/<dataset>/<run>`),
and writes its test predictions there: `predictions.jsonl` (the predicted distribution per
document) and `pred_results.json` (the metrics of `finetune.soft_label_metrics`). This
notebook recomputes everything from the predictions and checks it against that file. (For
a checkpoint instead, set `STEP`: the predictions are then read from `<run>-step<STEP>-val`
or `-test`, made with `train_classifier.sh` and the checkpoint as `--model_name`.)

**Measures** (all against the Qwen3.8-27B distribution of each document):

| Measure | Meaning | Better |
|---|---|---|
| Accuracy | The predicted top label is Qwen's top label | higher |
| Accuracy, confident | The same, on documents where Qwen's top label has probability > 0.5 | higher |
| Label-average accuracy | Accuracy (recall) per Qwen top label, averaged over labels: weighs rare labels equally | higher |
| Macro F1 | F1 per label of predicted vs Qwen top label, averaged over labels | higher |
| JSD (bits) | Jensen-Shannon divergence between the two distributions, 0 to 1 | lower |
| KL (bits) | KL divergence from Qwen's distribution to the prediction | lower |
| Brier | Summed squared difference of the distributions, 0 to 2 | lower |
| ECE | Expected calibration error of the predicted top label's probability (10 bins) | lower |

Confidence intervals are 95% normal approximations over documents. Differences between
the URL and no-URL model are **paired** (same documents), which makes them much tighter
than the intervals of the two models separately. Documents of the same language or source
are not independent, so treat the intervals as a guide rather than exact.

In [ ]:
from __future__ import annotations

import json
import math
from pathlib import Path

import datasets
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.compute as pc
from IPython.display import Markdown, display
from matplotlib.colors import LinearSegmentedColormap
from sklearn.metrics import f1_score

datasets.disable_progress_bars()
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# What to evaluate. SPLIT: "validation" or "test". STEP: a checkpoint step whose predictions
# are in <run>-step<STEP>-val (validation) or <run>-step<STEP>-test next to each run, or None
# for the final models (predictions.jsonl in the run directory, on the test split).
# EXCLUDE_LEAKED: leave out documents that are near or exact duplicates of training
# documents (from find_duplicates.py) in every section; section 11 compares both either way.
SPLIT = "test"
STEP = None
EXCLUDE_LEAKED = True
EPOCH_STEPS = 428174
LABEL_TYPES = ("topic", "format")
VARIANTS = ("url", "nourl")
VARIANT_NAMES = {"url": "URL + text", "nourl": "Text only"}
DATASETS = {"url": "hplt-finetune", "nourl": "hplt-finetune-nourl"}
RUNS = {
    ("topic", "url"): "mmbert-topic-1",
    ("topic", "nourl"): "mmbert-topic-nourl-1",
    ("format", "url"): "mmbert-format-1",
    ("format", "nourl"): "mmbert-format-nourl-1",
}
SOURCES = ("clean_random", "noisy_random", "poor_random")
DUPLICATES_DIR = REPO / "results/statistics/duplicates"
if SPLIT not in ("validation", "test"):
    raise ValueError(f"SPLIT must be 'validation' or 'test', got {SPLIT!r}")
if STEP is None and SPLIT != "test":
    raise ValueError("The final models' own predictions are on the test split; set SPLIT = 'test'")
# Shown in figure titles.
WHEN = (f"step {STEP:,}" if STEP is not None else "final models") + f", {SPLIT} split" \
    + (", duplicates of training excluded" if EXCLUDE_LEAKED else "")


def prediction_dir(label_type: str, variant: str) -> Path:
    # Directory with predictions.jsonl and pred_results.json for the configured model and split.
    run_dir = REPO / "results/models" / label_type / DATASETS[variant] / RUNS[label_type, variant]
    if STEP is None:
        return run_dir
    return run_dir.with_name(f"{run_dir.name}-step{STEP}-{'val' if SPLIT == 'validation' else 'test'}")


def dataset_dir(label_type: str, variant: str) -> Path:
    return REPO / "results/datasets" / label_type / DATASETS[variant]


# Colours (validated with the dataviz palette checker): blue = URL + text, orange = text only.
COLORS = {"url": "#2a78d6", "nourl": "#eb6834"}
NEUTRAL = "#8a8a86"
INK, MUTED, GRID = "#1f1f1e", "#5c5c58", "#e4e3df"
SEQUENTIAL = LinearSegmentedColormap.from_list("blues", ["#ffffff", "#cfe0f6", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
mpl.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.8, "axes.axisbelow": True, "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold", "legend.frameon": False,
    "figure.dpi": 100,
})
print(f"Evaluating: {WHEN}" + (f" ({STEP / EPOCH_STEPS:.2f} epochs)" if STEP is not None else ""))

## 1. Load the targets and predictions

The targets come from the evaluated split; the predictions are matched to them by
document ID. Every check fails loudly: the same documents in the URL and no-URL
datasets, identical target distributions, one prediction per document, and predicted
distributions over exactly the dataset's labels.

In [ ]:
def read_language_names(path: Path) -> dict[str, str]:
    # Dataset language key (name in lower case, spaces as underscores) -> display name.
    names = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            name, _code = (part.strip() for part in line.split(":", 1))
            names[name.lower().replace(" ", "_")] = name
    return names


def load_targets(label_type: str, variant: str) -> tuple[list[str], pd.DataFrame, np.ndarray]:
    # Label names, document columns, and target distributions of the evaluated split.
    path = dataset_dir(label_type, variant)
    label_names = json.loads((path / "labels.json").read_text(encoding="utf-8"))["label_names"]
    split = datasets.load_from_disk(str(path / SPLIT))
    table = split.select_columns(["id", "language", "source", "labels"]).with_format("arrow")[:]
    # Flattened in one go (much faster than to_pylist for the 364k-document test split).
    labels = table["labels"].combine_chunks()
    # Works for both list and fixed-size list columns.
    if labels.null_count or (pc.list_value_length(labels).to_numpy(zero_copy_only=False) != len(label_names)).any():
        raise ValueError(f"{path}: a label distribution is missing or has the wrong length")
    targets = labels.flatten().to_numpy().astype(np.float64).reshape(len(labels), len(label_names))
    docs = pd.DataFrame({name: table[name].to_pylist() for name in ("id", "language", "source")})
    if targets.shape != (len(docs), len(label_names)):
        raise ValueError(f"{path}: targets {targets.shape} for {len(docs)} documents and {len(label_names)} labels")
    if docs["id"].duplicated().any():
        raise ValueError(f"{path}: duplicate document IDs in the {SPLIT} split")
    return label_names, docs, targets


def load_predictions(label_type: str, variant: str, label_names: list[str], docs: pd.DataFrame) -> np.ndarray:
    # Predicted distributions in the order of `docs`, checked against it.
    path = prediction_dir(label_type, variant) / "predictions.jsonl"
    position = pd.Series(np.arange(len(docs)), index=docs["id"])
    probs = np.full((len(docs), len(label_names)), np.nan)
    seen = np.zeros(len(docs), dtype=bool)
    expected_labels = set(label_names)
    with path.open(encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            record = json.loads(line)
            if record["id"] not in position.index:
                raise ValueError(f"{path}:{line_number}: document {record['id']!r} is not in the {SPLIT} split")
            row = position[record["id"]]
            if seen[row]:
                raise ValueError(f"{path}:{line_number}: second prediction for {record['id']!r}")
            if set(record["label_probs"]) != expected_labels:
                raise ValueError(f"{path}:{line_number}: labels differ from the dataset's")
            if (record["language"], record["source"]) != (docs.at[row, "language"], docs.at[row, "source"]):
                raise ValueError(f"{path}:{line_number}: language or source differs from the dataset's")
            probs[row] = [record["label_probs"][name] for name in label_names]
            seen[row] = True
    if not seen.all():
        raise ValueError(f"{path}: no prediction for {int((~seen).sum())} documents")
    sums = probs.sum(axis=1)
    if not np.isfinite(probs).all() or (probs < 0).any() or np.abs(sums - 1).max() > 1e-4:
        raise ValueError(f"{path}: predicted distributions are not valid probabilities")
    return probs


LANGUAGE_NAMES = read_language_names(REPO / "data/qwen_label_languages.txt")
LABELS: dict[str, list[str]] = {}
DOCS: dict[str, pd.DataFrame] = {}
TARGETS: dict[str, np.ndarray] = {}
PROBS: dict[tuple[str, str], np.ndarray] = {}
for label_type in LABEL_TYPES:
    LABELS[label_type], DOCS[label_type], TARGETS[label_type] = load_targets(label_type, "url")
    nourl_labels, nourl_docs, nourl_targets = load_targets(label_type, "nourl")
    if nourl_labels != LABELS[label_type] or not nourl_docs.equals(DOCS[label_type]) \
            or not np.array_equal(nourl_targets, TARGETS[label_type]):
        raise ValueError(f"{label_type}: the URL and no-URL {SPLIT} splits differ")
    unknown = sorted(set(DOCS[label_type]["language"]) - set(LANGUAGE_NAMES))
    if unknown:
        raise ValueError(f"Languages missing from data/qwen_label_languages.txt: {unknown}")
    for variant in VARIANTS:
        PROBS[label_type, variant] = load_predictions(label_type, variant, LABELS[label_type], DOCS[label_type])
    print(f"{label_type}: {len(DOCS[label_type]):,} documents, {DOCS[label_type]['language'].nunique()} languages, "
          f"{len(LABELS[label_type])} labels; predictions of both models loaded")

if not DOCS["topic"]["id"].sort_values().reset_index(drop=True).equals(
        DOCS["format"]["id"].sort_values().reset_index(drop=True)):
    print(f"Note: the topic and format {SPLIT} splits hold different documents")
else:
    print(f"The topic and format {SPLIT} splits hold the same documents")

Training documents per language, for relating performance to the amount of training data.
They come from the per-language table of the dataset's `README.md` (written by
`create_datasets.py`), because reading the 30 GB training split over the mount is slow; the
table's column for the evaluated split is checked against the split loaded above.

In [ ]:
def read_language_counts(readme: Path) -> pd.DataFrame:
    # The "Documents per language" table of a dataset README: language -> documents per split.
    lines = readme.read_text(encoding="utf-8").splitlines()
    start = lines.index("### Documents per language")
    rows, header = {}, None
    for line in lines[start + 1:]:
        if line.startswith("#"):
            break
        if not line.startswith("|") or set(line) <= set("|-: "):
            continue
        cells = [cell.strip().strip("*") for cell in line.strip("|").split("|")]
        if header is None:
            header = cells
        elif cells[0] != "Total":
            rows[cells[0]] = [int(cell.replace(",", "")) for cell in cells[1:]]
    if header is None or not rows:
        raise ValueError(f"{readme}: no per-language table found")
    return pd.DataFrame.from_dict(rows, orient="index", columns=header[1:])


counts = read_language_counts(dataset_dir("topic", "url") / "README.md")
split_counts = DOCS["topic"]["language"].value_counts()
if set(counts.index) != set(split_counts.index) or \
        not (counts[SPLIT] == split_counts.reindex(counts.index)).all():
    raise ValueError(f"The README's per-language {SPLIT} counts don't match the {SPLIT} split")
TRAIN_DOCS = counts["train"].sort_index()
print(f"{TRAIN_DOCS.sum():,} training documents in {len(TRAIN_DOCS)} languages; "
      f"{(TRAIN_DOCS < TRAIN_DOCS.max()).sum()} languages have fewer than the full {TRAIN_DOCS.max():,}")

### Which checkpoint each final model is

`finetune.py` evaluates and checkpoints each run 20 times and, at the end of training, loads
the checkpoint with the lowest validation JSD and saves it as the run's model; the test
predictions come from that model. The table shows the step it was chosen from, and the curve
the validation JSD at every evaluation (from the run's `trainer_state.json`). If the best
step is the last one, the model was still improving when the epoch ended.

In [ ]:
def evaluation_history(run_dir: Path) -> tuple[dict[str, object], pd.DataFrame]:
    # The run's final trainer state and its evaluations during training (last entry per step).
    state = json.loads((run_dir / "trainer_state.json").read_text(encoding="utf-8"))
    evaluations = pd.DataFrame([entry for entry in state["log_history"] if "eval_js_divergence_bits" in entry])
    if evaluations.empty or state.get("best_global_step") is None:
        raise ValueError(f"{run_dir}/trainer_state.json has no evaluations or no best checkpoint")
    return state, evaluations.drop_duplicates("step", keep="last").sort_values("step")


if STEP is None:
    rows, histories = [], {}
    for label_type in LABEL_TYPES:
        for variant in VARIANTS:
            state, history = evaluation_history(prediction_dir(label_type, variant))
            histories[label_type, variant] = history
            best = history[history["step"] == state["best_global_step"]]
            if len(best) != 1 or not np.isclose(best["eval_js_divergence_bits"].iloc[0], state["best_metric"]):
                raise ValueError(f"{label_type}/{variant}: the best step's evaluation doesn't match best_metric")
            rows.append({
                "Label type": label_type, "Input": VARIANT_NAMES[variant],
                "Steps trained": state["global_step"], "Evaluations": len(history),
                "Best step": state["best_global_step"], "Last step": int(history["step"].iloc[-1]),
                "Best validation JSD": state["best_metric"],
                "Last validation JSD": history["eval_js_divergence_bits"].iloc[-1],
                "Validation accuracy at best": best["eval_accuracy"].iloc[0],
            })
    display(pd.DataFrame(rows).style.format({"Steps trained": "{:,}", "Best step": "{:,}", "Last step": "{:,}",
                                             "Best validation JSD": "{:.4f}", "Last validation JSD": "{:.4f}",
                                             "Validation accuracy at best": "{:.1%}"}).hide(axis="index"))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
    for ax, label_type in zip(axes, LABEL_TYPES):
        for variant in VARIANTS:
            history = histories[label_type, variant]
            ax.plot(history["step"] / EPOCH_STEPS, history["eval_js_divergence_bits"], marker="o", ms=4, lw=2,
                    color=COLORS[variant], label=VARIANT_NAMES[variant])
            best = history.loc[history["eval_js_divergence_bits"].idxmin()]
            ax.scatter(best["step"] / EPOCH_STEPS, best["eval_js_divergence_bits"], s=90, facecolor="white",
                       edgecolor=COLORS[variant], linewidth=2, zorder=3)
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Validation JSD (bits)")
        ax.set_title(f"{label_type.capitalize()} (ring: best evaluation)")
        ax.legend(loc="upper right")
    fig.suptitle("Validation JSD during training", fontweight="bold")
    plt.show()
else:
    print(f"STEP = {STEP}: checkpoint predictions, so no best-checkpoint selection is involved")

## 2. Per-document measures, checked against the run's own metrics

Each document gets its correctness, divergences and Brier score once; every table below
averages these. The overall, per-language and per-source values are compared with
`pred_results.json` (which `finetune.py` computed from the logits; the predictions file
stores rounded probabilities, hence a small tolerance).

In [ ]:
LN2 = math.log(2)


def safe_log(x: np.ndarray) -> np.ndarray:
    # log(x), with 0 for x == 0 (those terms are multiplied by 0).
    return np.log(np.where(x > 0, x, 1.0))


def per_document(label_type: str, variant: str) -> pd.DataFrame:
    probs, targets = PROBS[label_type, variant], TARGETS[label_type]
    log_probs, log_targets = safe_log(probs), safe_log(targets)
    mixture = (probs + targets) / 2
    log_mixture = safe_log(mixture)
    frame = DOCS[label_type][["language", "source"]].copy()
    frame["target"] = targets.argmax(axis=1)
    frame["predicted"] = probs.argmax(axis=1)
    frame["correct"] = frame["target"] == frame["predicted"]
    frame["target_confidence"] = targets.max(axis=1)
    frame["predicted_confidence"] = probs.max(axis=1)
    frame["kl"] = (targets * (log_targets - log_probs)).sum(axis=1) / LN2
    frame["js"] = (0.5 * (targets * (log_targets - log_mixture)).sum(axis=1)
                   + 0.5 * (probs * (log_probs - log_mixture)).sum(axis=1)) / LN2
    frame["brier"] = ((probs - targets) ** 2).sum(axis=1)
    if not np.isfinite(frame[["kl", "js", "brier"]].to_numpy()).all():
        raise ValueError(f"{label_type}/{variant}: non-finite divergence (a predicted probability of 0?)")
    return frame


def expected_calibration_error(frame: pd.DataFrame, bins: int = 10) -> float:
    edges = np.linspace(0, 1, bins + 1)
    index = np.clip(np.digitize(frame["predicted_confidence"], edges[1:-1]), 0, bins - 1)
    grouped = frame.groupby(index)
    gaps = (grouped["predicted_confidence"].mean() - grouped["correct"].mean()).abs()
    return float((gaps * grouped.size() / len(frame)).sum())


def summarise(frame: pd.DataFrame, n_labels: int) -> dict[str, float]:
    present = sorted(frame["target"].unique())
    recall = frame.groupby("target")["correct"].mean()
    confident = frame["target_confidence"] > 0.5
    return {
        "Documents": len(frame),
        "Accuracy": frame["correct"].mean(),
        "Accuracy, confident": frame.loc[confident, "correct"].mean(),
        "Label-average accuracy": recall.mean(),
        "Macro F1": f1_score(frame["target"], frame["predicted"], labels=present, average="macro", zero_division=0),
        "JSD (bits)": frame["js"].mean(),
        "KL (bits)": frame["kl"].mean(),
        "Brier": frame["brier"].mean(),
        "ECE": expected_calibration_error(frame),
    }


FRAMES = {(label_type, variant): per_document(label_type, variant)
          for label_type in LABEL_TYPES for variant in VARIANTS}

# Check against pred_results.json: overall, per language, per source.
CHECKS = {"accuracy": "Accuracy", "accuracy_label_average": "Label-average accuracy",
          "js_divergence_bits": "JSD (bits)", "kl_divergence_bits": "KL (bits)", "brier_score": "Brier"}
worst = 0.0
for (label_type, variant), frame in FRAMES.items():
    saved = json.loads((prediction_dir(label_type, variant) / "pred_results.json").read_text(encoding="utf-8"))
    groups = {"test_": frame}
    groups |= {f"test_{language}_": part for language, part in frame.groupby("language")}
    groups |= {f"test_source_{source}_": part for source, part in frame.groupby("source")}
    for prefix, part in groups.items():
        mine = summarise(part, len(LABELS[label_type]))
        for key, name in CHECKS.items():
            if prefix + key not in saved:
                raise KeyError(f"{label_type}/{variant}: {prefix + key} missing from pred_results.json")
            difference = abs(mine[name] - saved[prefix + key])
            worst = max(worst, difference / max(abs(saved[prefix + key]), 1e-3))
            if difference > 1e-4 + 1e-3 * abs(saved[prefix + key]):
                raise ValueError(f"{label_type}/{variant} {prefix + key}: {mine[name]} here, {saved[prefix + key]} saved")
print(f"All overall, per-language and per-source metrics match pred_results.json (largest relative difference {worst:.1e})")

### Documents duplicated from the training split

`scripts/data_analysis/find_duplicates.py` (`hpc/roihu/find_duplicates.sh <label type>/hplt-finetune`)
lists the documents of the evaluated split whose text is a near or exact duplicate of a
training document (`duplicates.tsv`, `leak` = 1; character 5-gram MinHash, estimated
Jaccard similarity ≥ 0.8), and those whose URL also occurs in the training split
(`url_duplicates.tsv`). With `EXCLUDE_LEAKED = True` the duplicated documents are left out
of every section below, as if the evaluated split had been deduplicated against training.
Documents that only share a URL with training are kept; section 11 checks that the URL
models don't profit from them, and compares the results with and without the duplicated
documents.

In [ ]:
def read_duplicates(label_type: str) -> tuple[np.ndarray, pd.DataFrame]:
    # Rows of the evaluated split duplicated from training, and its documents whose URL is in training
    # (position, text similarity to the first training copy, that copy's top label).
    directory = DUPLICATES_DIR / label_type / DATASETS["url"]
    summary = json.loads((directory / "summary.json").read_text(encoding="utf-8"))
    if summary["documents"].get(SPLIT) != len(DOCS[label_type]):
        raise ValueError(f"{directory}: {summary['documents'].get(SPLIT)} {SPLIT} documents, "
                         f"the dataset has {len(DOCS[label_type])}; rerun find_duplicates.py")
    position = pd.Series(np.arange(len(DOCS[label_type])), index=DOCS[label_type]["id"])
    clusters = pd.read_csv(directory / "duplicates.tsv", sep="\t", dtype=str, keep_default_na=False)
    leaked_ids = clusters.loc[(clusters["split"] == SPLIT) & (clusters["leak"] == "1"), "id"]
    if not leaked_ids.isin(position.index).all():
        raise ValueError(f"{directory}/duplicates.tsv lists {SPLIT} documents that aren't in the split")
    leaked = np.zeros(len(position), dtype=bool)
    leaked[position[leaked_ids].to_numpy()] = True

    urls = pd.read_csv(directory / "url_duplicates.tsv", sep="\t", dtype=str, keep_default_na=False)
    train = urls[urls["split"] == "train"]
    # Groups are ordered training split first, so a leaked document's first copy is a training document.
    first_split = urls.groupby("url_group")["split"].first()
    twin_label = train.groupby("url_group")["top_label"].agg(lambda labels: labels.value_counts().index[0])
    repeats = urls[(urls["split"] == SPLIT) & (urls["leak"] == "1")]
    if (first_split[repeats["url_group"]] != "train").any() or not repeats["url_group"].isin(twin_label.index).all():
        raise ValueError(f"{directory}/url_duplicates.tsv: a URL leak without a first training copy")
    if not repeats["id"].isin(position.index).all():
        raise ValueError(f"{directory}/url_duplicates.tsv lists {SPLIT} documents that aren't in the split")
    url_repeats = pd.DataFrame({
        "position": position[repeats["id"]].to_numpy(),
        "similarity": pd.to_numeric(repeats["estimated_jaccard_to_first"].replace("", np.nan)).to_numpy(),
        "twin_label": [LABELS[label_type].index(name) for name in twin_label[repeats["url_group"]]],
    })
    return leaked, url_repeats


LEAKED, URL_REPEATS = {}, {}
for label_type in LABEL_TYPES:
    LEAKED[label_type], URL_REPEATS[label_type] = read_duplicates(label_type)
    print(f"{label_type}: {LEAKED[label_type].sum():,} of {len(LEAKED[label_type]):,} {SPLIT} documents "
          f"({LEAKED[label_type].mean():.2%}) duplicate a training document; "
          f"{len(URL_REPEATS[label_type]):,} share a URL with one")

# Everything before exclusion, for section 11.
DOCS_ALL, TARGETS_ALL, PROBS_ALL, FRAMES_ALL = dict(DOCS), dict(TARGETS), dict(PROBS), dict(FRAMES)
if EXCLUDE_LEAKED:
    for label_type in LABEL_TYPES:
        keep = ~LEAKED[label_type]
        DOCS[label_type] = DOCS[label_type][keep].reset_index(drop=True)
        TARGETS[label_type] = TARGETS[label_type][keep]
        for variant in VARIANTS:
            PROBS[label_type, variant] = PROBS[label_type, variant][keep]
            FRAMES[label_type, variant] = FRAMES[label_type, variant][keep].reset_index(drop=True)
    print("Duplicated documents are excluded from the following sections")
else:
    print("All documents are kept (EXCLUDE_LEAKED = False); section 11 shows the results without the duplicated ones")

## 3. Overall results

The best value of each measure within a label type is in bold.

In [ ]:
HIGHER_IS_BETTER = {"Accuracy", "Accuracy, confident", "Label-average accuracy", "Macro F1"}

overall = pd.DataFrame(
    {(label_type, VARIANT_NAMES[variant]): summarise(FRAMES[label_type, variant], len(LABELS[label_type]))
     for label_type in LABEL_TYPES for variant in VARIANTS}
).T
overall.index.names = ["Label type", "Input"]


def bold_best(table: pd.DataFrame, group_level: str | None = "Label type"):
    # Styler that bolds the best value of each measure (within each group of rows).
    def column_styles(column: pd.Series) -> list[str]:
        if column.name not in HIGHER_IS_BETTER and column.name not in LOWER_IS_BETTER:
            return [""] * len(column)
        groups = column.groupby(level=group_level) if group_level else [(None, column)]
        best = pd.Series(False, index=column.index)
        for _, part in groups:
            target = part.max() if column.name in HIGHER_IS_BETTER else part.min()
            best[part.index] = np.isclose(part, target, rtol=0, atol=1e-12)
        return ["font-weight: bold" if flag else "" for flag in best]
    return table.style.apply(column_styles)


LOWER_IS_BETTER = {"JSD (bits)", "KL (bits)", "Brier", "ECE"}
formats = {name: "{:.3f}" for name in overall.columns}
formats["Documents"] = "{:,.0f}"
display(bold_best(overall).format(formats))

### The URL effect overall

The paired difference URL + text − text only on the same documents, with a 95% interval.
Positive accuracy and negative JSD differences favour the URL.

In [ ]:
def paired_difference(a: pd.Series, b: pd.Series) -> tuple[float, float, float]:
    # Mean of a - b with a 95% normal interval (a and b aligned on the same documents).
    difference = a.to_numpy(dtype=float) - b.to_numpy(dtype=float)
    mean = difference.mean()
    half = 1.96 * difference.std(ddof=1) / math.sqrt(len(difference)) if len(difference) > 1 else float("nan")
    return mean, mean - half, mean + half


rows = []
for label_type in LABEL_TYPES:
    url, nourl = FRAMES[label_type, "url"], FRAMES[label_type, "nourl"]
    for measure, column, scale, unit in (("Accuracy", "correct", 100, "pp"), ("JSD", "js", 1, "bits")):
        mean, low, high = paired_difference(url[column], nourl[column])
        rows.append({"Label type": label_type, "Measure": f"{measure} ({unit})",
                     "URL − text only": mean * scale, "95% low": low * scale, "95% high": high * scale})
url_effect = pd.DataFrame(rows).set_index(["Label type", "Measure"])
display(url_effect.style.format({"URL − text only": "{:+.3f}", "95% low": "{:+.3f}", "95% high": "{:+.3f}"}))
for label_type in LABEL_TYPES:
    changed = FRAMES[label_type, "url"]["predicted"] != FRAMES[label_type, "nourl"]["predicted"]
    print(f"{label_type}: the two models give a different top label for {changed.mean():.1%} of the documents")

## 4. By source

The evaluated split mixes the three HPLT samples in the proportions of training (40% clean,
42% noisy, 18% poor). Error bars are 95% intervals.

In [ ]:
by_source = pd.DataFrame(
    {(label_type, VARIANT_NAMES[variant], source): summarise(part, len(LABELS[label_type]))
     for label_type in LABEL_TYPES for variant in VARIANTS
     for source, part in FRAMES[label_type, variant].groupby("source")}
).T
by_source.index.names = ["Label type", "Input", "Source"]
by_source = by_source.reorder_levels(["Label type", "Source", "Input"]).reindex(
    [(label_type, source, VARIANT_NAMES[variant]) for label_type in LABEL_TYPES for source in SOURCES
     for variant in VARIANTS])
display(bold_best(by_source[["Documents", "Accuracy", "Label-average accuracy", "Macro F1", "JSD (bits)", "Brier"]],
                  group_level=["Label type", "Source"]).format({**formats}))

fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True, layout="constrained")
for column, label_type in enumerate(LABEL_TYPES):
    for row, (measure, field, scale) in enumerate((("Accuracy (%)", "correct", 100), ("JSD (bits)", "js", 1))):
        ax = axes[row, column]
        for offset, variant in zip((-0.12, 0.12), VARIANTS):
            frame = FRAMES[label_type, variant]
            stats = frame.groupby("source")[field].agg(["mean", "std", "size"]).loc[list(SOURCES)]
            half = 1.96 * stats["std"] / np.sqrt(stats["size"])
            x = np.arange(len(SOURCES)) + offset
            ax.errorbar(x, stats["mean"] * scale, yerr=half * scale, fmt="o", ms=8, color=COLORS[variant],
                        ecolor=COLORS[variant], elinewidth=2, capsize=0, label=VARIANT_NAMES[variant])
            for xi, value in zip(x, stats["mean"] * scale):
                ax.annotate(f"{value:.1f}" if scale == 100 else f"{value:.3f}", (xi, value), xytext=(8 if offset > 0 else -8, 0),
                            textcoords="offset points", ha="left" if offset > 0 else "right", va="center",
                            fontsize=8, color=MUTED)
        ax.set_xticks(range(len(SOURCES)), [source.replace("_random", "") for source in SOURCES])
        ax.set_xlim(-0.6, len(SOURCES) - 0.4)
        ax.set_ylabel(measure)
        ax.grid(axis="x", visible=False)
        if row == 0:
            ax.set_title(label_type.capitalize())
axes[0, 0].legend(loc="lower left")
fig.suptitle(f"Accuracy and JSD by HPLT source ({WHEN})", fontweight="bold")
plt.show()

## 5. By language

Accuracy per language for both inputs, sorted by the URL model's accuracy. Each language
has about 600 validation documents (3,000 test documents), so a language's accuracy has a 95%
interval of roughly ±3.5 points on validation (±1.5 on test); differences of a few points
between neighbours are noise. Languages marked
with * have fewer training documents than the full 56,400.

In [ ]:
def language_table(label_type: str) -> pd.DataFrame:
    url, nourl = FRAMES[label_type, "url"], FRAMES[label_type, "nourl"]
    rows = {}
    for language in sorted(url["language"].unique()):
        a, b = url[url["language"] == language], nourl[nourl["language"] == language]
        mean, low, high = paired_difference(a["correct"], b["correct"])
        rows[language] = {
            "Language": LANGUAGE_NAMES[language],
            "Train docs": int(TRAIN_DOCS[language]),
            "Evaluated docs": len(a),
            "Accuracy, URL": a["correct"].mean(),
            "Accuracy, text only": b["correct"].mean(),
            "URL effect (pp)": 100 * mean, "Effect low": 100 * low, "Effect high": 100 * high,
            "JSD, URL": a["js"].mean(),
            "JSD, text only": b["js"].mean(),
            "Qwen confidence": a["target_confidence"].mean(),
        }
    return pd.DataFrame.from_dict(rows, orient="index")


LANGUAGE_TABLES = {label_type: language_table(label_type) for label_type in LABEL_TYPES}
FULL_TRAIN = TRAIN_DOCS.max()

fig, axes = plt.subplots(1, 2, figsize=(13, 0.165 * len(TRAIN_DOCS) + 1.5), layout="constrained")
for ax, label_type in zip(axes, LABEL_TYPES):
    table = LANGUAGE_TABLES[label_type].sort_values("Accuracy, URL")
    y = np.arange(len(table))
    ax.hlines(y, table["Accuracy, text only"] * 100, table["Accuracy, URL"] * 100, color=GRID, lw=2, zorder=1)
    for variant, column in (("nourl", "Accuracy, text only"), ("url", "Accuracy, URL")):
        ax.scatter(table[column] * 100, y, s=22, color=COLORS[variant], edgecolor="white", linewidth=0.8,
                   zorder=2, label=VARIANT_NAMES[variant])
    labels = [name + (" *" if docs < FULL_TRAIN else "") for name, docs in zip(table["Language"], table["Train docs"])]
    ax.set_yticks(y, labels, fontsize=7.5)
    ax.set_ylim(-0.8, len(table) - 0.2)
    ax.grid(axis="y", visible=False)
    ax.set_xlabel("Accuracy against Qwen3.8-27B's top label (%)")
    ax.set_title(f"{label_type.capitalize()}")
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles[::-1], labels[::-1], loc="lower right")
fig.suptitle(f"Accuracy per language ({WHEN}; * = fewer training documents)", fontweight="bold")
plt.show()

In [ ]:
for label_type in LABEL_TYPES:
    table = LANGUAGE_TABLES[label_type]
    accuracy = table["Accuracy, URL"]
    display(Markdown(
        f"**{label_type.capitalize()}, URL model:** median language accuracy {accuracy.median():.1%}, "
        f"range {accuracy.min():.1%} ({table.loc[accuracy.idxmin(), 'Language']}) to "
        f"{accuracy.max():.1%} ({table.loc[accuracy.idxmax(), 'Language']}); "
        f"interquartile range {accuracy.quantile(0.25):.1%}–{accuracy.quantile(0.75):.1%}."))
    columns = ["Language", "Train docs", "Accuracy, URL", "Accuracy, text only", "URL effect (pp)", "JSD, URL",
               "Qwen confidence"]
    style = {"Train docs": "{:,}", "Accuracy, URL": "{:.1%}", "Accuracy, text only": "{:.1%}",
             "URL effect (pp)": "{:+.1f}", "JSD, URL": "{:.3f}", "Qwen confidence": "{:.2f}"}
    display(Markdown(f"Ten lowest-accuracy languages ({label_type}, URL model):"))
    display(table.sort_values("Accuracy, URL")[columns].head(10).style.format(style).hide(axis="index"))

### What makes a language hard?

Two candidate explanations: less training data (the 17 smaller languages) and more
ambiguous targets. *Qwen confidence* is the mean probability of Qwen3.8-27B's top
label in that language: low values mean the labeller itself spread its probability over
several labels, so "the" top label is less well defined and harder to hit.

In [ ]:
from scipy.stats import spearmanr

fig, axes = plt.subplots(1, 2, figsize=(12, 5), layout="constrained")
for ax, label_type in zip(axes, LABEL_TYPES):
    table = LANGUAGE_TABLES[label_type]
    small = table["Train docs"] < FULL_TRAIN
    ax.scatter(table.loc[~small, "Qwen confidence"], table.loc[~small, "Accuracy, URL"] * 100, s=30,
               color=COLORS["url"], edgecolor="white", linewidth=0.8, label=f"{FULL_TRAIN:,} training documents")
    ax.scatter(table.loc[small, "Qwen confidence"], table.loc[small, "Accuracy, URL"] * 100, s=34,
               facecolor="white", edgecolor=COLORS["url"], linewidth=1.6, label="Fewer training documents")
    rho, p = spearmanr(table["Qwen confidence"], table["Accuracy, URL"])
    for language in table["Accuracy, URL"].nsmallest(5).index.union(table["Accuracy, URL"].nlargest(3).index):
        ax.annotate(table.at[language, "Language"], (table.at[language, "Qwen confidence"],
                    table.at[language, "Accuracy, URL"] * 100), xytext=(5, 0), textcoords="offset points",
                    fontsize=8, color=MUTED, va="center")
    ax.set_xlabel("Mean probability of Qwen's top label in the language")
    ax.set_ylabel("Accuracy, URL model (%)")
    ax.set_title(f"{label_type.capitalize()} (Spearman ρ = {rho:.2f})")
    ax.legend(loc="upper left")
fig.suptitle("Per-language accuracy against how confident the labeller was", fontweight="bold")
plt.show()

for label_type in LABEL_TYPES:
    table = LANGUAGE_TABLES[label_type]
    small = table["Train docs"] < FULL_TRAIN
    display(Markdown(
        f"**{label_type.capitalize()}:** mean accuracy {table.loc[~small, 'Accuracy, URL'].mean():.1%} in the "
        f"{(~small).sum()} full languages, {table.loc[small, 'Accuracy, URL'].mean():.1%} in the {small.sum()} smaller "
        f"ones (mean Qwen confidence {table.loc[~small, 'Qwen confidence'].mean():.2f} vs "
        f"{table.loc[small, 'Qwen confidence'].mean():.2f}). Spearman correlation of training documents and "
        f"accuracy among the smaller languages: {spearmanr(table.loc[small, 'Train docs'], table.loc[small, 'Accuracy, URL'])[0]:.2f}."))

both = LANGUAGE_TABLES["topic"][["Accuracy, URL"]].join(LANGUAGE_TABLES["format"][["Accuracy, URL"]],
                                                        lsuffix=" topic", rsuffix=" format")
display(Markdown(f"Languages that are hard for topic tend to be hard for format too: Spearman ρ between the two "
                 f"per-language accuracies is {spearmanr(both.iloc[:, 0], both.iloc[:, 1])[0]:.2f}."))

## 6. The URL effect per language

The paired accuracy difference (URL + text − text only) per language with its 95%
interval. Blue: the URL model is better beyond the interval; orange: the text-only model
is; gray: no clear difference. With 95% intervals and 128 languages, about 3 languages
would land clearly on each side of zero by chance alone if the URL had no effect at all.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 0.165 * len(TRAIN_DOCS) + 1.5), layout="constrained")
for ax, label_type in zip(axes, LABEL_TYPES):
    table = LANGUAGE_TABLES[label_type].sort_values("URL effect (pp)")
    y = np.arange(len(table))
    color = np.where(table["Effect low"] > 0, COLORS["url"], np.where(table["Effect high"] < 0, COLORS["nourl"], NEUTRAL))
    ax.hlines(y, table["Effect low"], table["Effect high"], color=color, lw=1.6, alpha=0.6)
    ax.scatter(table["URL effect (pp)"], y, s=20, color=color, zorder=2)
    ax.axvline(0, color=MUTED, lw=1)
    overall_mean = 100 * paired_difference(FRAMES[label_type, "url"]["correct"], FRAMES[label_type, "nourl"]["correct"])[0]
    ax.axvline(overall_mean, color=COLORS["url"], lw=1, ls="--")
    ax.annotate(f"overall {overall_mean:+.1f}", (overall_mean, len(table) - 0.5), xytext=(4, 0),
                textcoords="offset points", fontsize=8, color=MUTED, va="top")
    ax.set_yticks(y, table["Language"], fontsize=7.5)
    ax.set_ylim(-0.8, len(table) - 0.2)
    ax.grid(axis="y", visible=False)
    ax.set_xlabel("Accuracy difference, URL + text − text only (percentage points)")
    better, worse = (table["Effect low"] > 0).sum(), (table["Effect high"] < 0).sum()
    ax.set_title(f"{label_type.capitalize()}: URL better in {better}, worse in {worse} languages")
fig.suptitle(f"Effect of the URL on accuracy per language ({WHEN})", fontweight="bold")
plt.show()

effects = LANGUAGE_TABLES["topic"][["URL effect (pp)"]].join(LANGUAGE_TABLES["format"][["URL effect (pp)"]],
                                                             lsuffix=", topic", rsuffix=", format")
display(Markdown(f"Correlation of the URL effect between topic and format across languages: Spearman ρ = "
                 f"{spearmanr(effects.iloc[:, 0], effects.iloc[:, 1])[0]:.2f}."))

## 7. By label

Per label: *recall* is the share of documents with that Qwen top label that the model also
gives as top label (the per-label accuracy of `finetune.py`); *precision* is the share of the
model's predictions of that label that agree with Qwen; *F1* combines them. *Support* is the
number of evaluated documents with that Qwen top label.

In [ ]:
def label_table(label_type: str) -> pd.DataFrame:
    names = LABELS[label_type]
    rows = {}
    for index, name in enumerate(names):
        row = {"Support": int((FRAMES[label_type, "url"]["target"] == index).sum())}
        for variant in VARIANTS:
            frame = FRAMES[label_type, variant]
            true_positive = ((frame["target"] == index) & (frame["predicted"] == index)).sum()
            predicted, actual = (frame["predicted"] == index).sum(), (frame["target"] == index).sum()
            precision = true_positive / predicted if predicted else float("nan")
            recall = true_positive / actual if actual else float("nan")
            suffix = ", URL" if variant == "url" else ", text only"
            row[f"Recall{suffix}"] = recall
            row[f"Precision{suffix}"] = precision
            row[f"F1{suffix}"] = (2 * precision * recall / (precision + recall)
                                  if precision + recall > 0 else 0.0)
        rows[name] = row
    return pd.DataFrame.from_dict(rows, orient="index")


LABEL_TABLES = {label_type: label_table(label_type) for label_type in LABEL_TYPES}

fig, axes = plt.subplots(1, 2, figsize=(13, 7.5), layout="constrained")
for ax, label_type in zip(axes, LABEL_TYPES):
    table = LABEL_TABLES[label_type].sort_values("F1, URL")
    y = np.arange(len(table))
    ax.hlines(y, table["F1, text only"] * 100, table["F1, URL"] * 100, color=GRID, lw=2, zorder=1)
    for variant, column in (("nourl", "F1, text only"), ("url", "F1, URL")):
        ax.scatter(table[column] * 100, y, s=30, color=COLORS[variant], edgecolor="white", linewidth=0.8,
                   zorder=2, label=VARIANT_NAMES[variant])
    ax.set_yticks(y, [f"{name} ({support:,})" for name, support in zip(table.index, table["Support"])], fontsize=8.5)
    ax.set_ylim(-0.8, len(table) - 0.2)
    ax.grid(axis="y", visible=False)
    ax.set_xlabel("F1 against Qwen's top label (%)")
    ax.set_title(f"{label_type.capitalize()} (support in brackets)")
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles[::-1], labels[::-1], loc="lower right")
fig.suptitle(f"Per-label F1 ({WHEN})", fontweight="bold")
plt.show()

for label_type in LABEL_TYPES:
    display(Markdown(f"**{label_type.capitalize()}**, sorted by support:"))
    table = LABEL_TABLES[label_type].sort_values("Support", ascending=False)
    display(table.style.format({column: "{:.1%}" for column in table.columns if column != "Support"} | {"Support": "{:,}"}))

## 8. Confusions

Row-normalised confusion matrices of the URL models: each row is a Qwen top label, and
the cells show where the model's top label went. The diagonal is the recall of section 7.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 8.2), layout="constrained")
for ax, label_type in zip(axes, LABEL_TYPES):
    names, frame = LABELS[label_type], FRAMES[label_type, "url"]
    matrix = pd.crosstab(frame["target"], frame["predicted"]).reindex(index=range(len(names)), columns=range(len(names)),
                                                                     fill_value=0).to_numpy(dtype=float)
    shares = matrix / np.maximum(matrix.sum(axis=1, keepdims=True), 1)
    mesh = ax.pcolormesh(shares, cmap=SEQUENTIAL, vmin=0, vmax=1, edgecolors="white", linewidth=0.5)
    ax.set_xticks(np.arange(len(names)) + 0.5, names, rotation=90, fontsize=8)
    ax.set_yticks(np.arange(len(names)) + 0.5, names, fontsize=8)
    ax.invert_yaxis()
    ax.grid(False)
    ax.set_xlabel("Model's top label")
    ax.set_ylabel("Qwen3.8-27B's top label")
    ax.set_title(label_type.capitalize())
    ax.set_aspect("equal")
fig.colorbar(mesh, ax=axes, shrink=0.6, label="Share of the row")
plt.show()

for label_type in LABEL_TYPES:
    names, frame = LABELS[label_type], FRAMES[label_type, "url"]
    wrong = frame[~frame["correct"]]
    pairs = wrong.groupby(["target", "predicted"]).size().sort_values(ascending=False).head(12)
    support = frame["target"].value_counts()
    table = pd.DataFrame({
        "Qwen's label": [names[t] for t, _ in pairs.index],
        "Model's label": [names[p] for _, p in pairs.index],
        "Documents": pairs.to_numpy(),
        "Share of Qwen's label": [count / support[t] for (t, _), count in pairs.items()],
        "Share of all errors": pairs.to_numpy() / len(wrong),
    })
    display(Markdown(f"**{label_type.capitalize()}:** the 12 most common confusions of the URL model "
                     f"({len(wrong):,} errors in all)"))
    display(table.style.format({"Documents": "{:,}", "Share of Qwen's label": "{:.1%}",
                                "Share of all errors": "{:.1%}"}).hide(axis="index"))

## 9. Calibration

How well the model's confidence in its top label matches how often that label agrees with
Qwen's top label. Points on the diagonal are perfectly calibrated; points below it are
over-confident, points above it under-confident. Bins with fewer than 100 documents are left out. Note that "correct" here
means agreeing with Qwen's single top label, while the model was trained on the whole
distribution: on documents where Qwen itself is split, even a perfect copy of Qwen's
distribution would look over-confident by this measure.

In [ ]:
edges = np.linspace(0, 1, 11)
fig, axes = plt.subplots(1, 2, figsize=(11, 5), layout="constrained")
for ax, label_type in zip(axes, LABEL_TYPES):
    ax.plot([0, 1], [0, 1], color=MUTED, lw=1, ls="--")
    for variant in VARIANTS:
        frame = FRAMES[label_type, variant]
        index = np.clip(np.digitize(frame["predicted_confidence"], edges[1:-1]), 0, 9)
        stats = frame.groupby(index).agg(confidence=("predicted_confidence", "mean"), accuracy=("correct", "mean"),
                                         size=("correct", "size"))
        stats = stats[stats["size"] >= 100]
        ax.plot(stats["confidence"], stats["accuracy"], marker="o", ms=7, lw=2, color=COLORS[variant],
                label=f"{VARIANT_NAMES[variant]} (ECE {expected_calibration_error(frame):.3f})")
    # Where the documents are: most have a confident top label.
    shares = np.histogram(FRAMES[label_type, "url"]["predicted_confidence"], bins=edges)[0] / len(FRAMES[label_type, "url"])
    ax.bar(edges[:-1] + 0.05, shares, width=0.09, color=GRID, zorder=0, label="Share of documents (URL model)")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel("Model's probability of its top label")
    ax.set_ylabel("Agreement with Qwen's top label")
    ax.set_title(label_type.capitalize())
    ax.legend(loc="upper left", fontsize=8.5)
fig.suptitle(f"Calibration ({WHEN})", fontweight="bold")
plt.show()

for label_type in LABEL_TYPES:
    targets = FRAMES[label_type, "url"]["target_confidence"]
    display(Markdown(
        f"**{label_type.capitalize()}:** mean probability of the top label: model (URL) "
        f"{FRAMES[label_type, 'url']['predicted_confidence'].mean():.3f}, text only "
        f"{FRAMES[label_type, 'nourl']['predicted_confidence'].mean():.3f}, Qwen3.8-27B {targets.mean():.3f}."))

## 10. Comparison with the original WebOrganizer

WebOrganizer ([Wettig et al., 2025](https://arxiv.org/abs/2502.10341), Appendix B, Table 7)
reports how well its classifiers imitate Llama-3.1-405B-Instruct on its validation set of
10K English web pages. It scores only the documents where **the labelling LLM's** top
category has at least 75% probability (86% of topic and 79% of format annotations), and
reports the accuracy (recall) per label averaged over the labels, and that of the worst
label. The threshold is on the labeller's confidence, not the classifier's: in
[`define_domains/train_classifier.py`](https://github.com/CodeCreator/WebOrganizer/blob/main/define_domains/train_classifier.py)
the subset is `np.max(labels, axis=-1) > 0.75`, the same as `accuracy_label_average_conf75`
and `accuracy_label_min_conf75` in `finetune.py`. Here Qwen3.8-27B is the labeller.

| WebOrganizer (Table 7) | Topic, label average | Topic, worst label | Format, label average | Format, worst label |
|---|---:|---:|---:|---:|
| With URL | 93.5% | 87.1% | 91.8% | 80.5% |
| Without URL | 92.1% | 86.0% | 88.9% | 80.2% |

The setups differ in ways that matter for these numbers:

| | WebOrganizer | This project |
|---|---|---|
| Labeller (targets) | Llama-3.1-405B-Instruct | Qwen3.8-27B |
| Labels | 24 topics, 24 formats | the same 24 + *Other* and *Bad example* (26) |
| Languages | English | 128 languages of HPLT 4.0, including noisy and poor-quality pages |
| Classifier | gte-base-en-v1.5 (140M parameters) | mmBERT-base |
| Training | 1M Llama-3.1-8B annotations, then 100K 405B annotations, 5 epochs each | 6.85M Qwen annotations, one epoch |
| Evaluation | 10K validation pages | 362,706 test pages, about 3,000 per language, without duplicates of training documents |

So the comparison below is indicative only. To bring it closer, our models are also scored
on WebOrganizer's **24 shared labels**: documents whose Qwen top label is *Other* or *Bad
example* are left out, and the model's prediction is its most probable label among the 24,
as if it had been trained without the two extra labels. *English only* restricts that to the
about 3,000 English test documents. For English the worst label is not reported, and the
label average covers only the labels with at least 20 confident English documents (*Labels*
column), so it is comparable with the 24-label averages only if that column says 24. *Accuracy* (the share of all confident documents whose top label is right, which
WebOrganizer doesn't report) compares English with all languages fairly.

The worst label is the minimum of many noisy per-label estimates, so it is biased downwards
and its interval is rough, especially for labels with few confident documents.

Intervals are 95% bootstrap intervals over documents (1,000 resamples).

In [ ]:
# WebOrganizer, Table 7 (Wettig et al., 2025, Appendix B): (label average, worst label).
WEBORGANIZER = {
    ("topic", "url"): (0.935, 0.871), ("topic", "nourl"): (0.921, 0.860),
    ("format", "url"): (0.918, 0.805), ("format", "nourl"): (0.889, 0.802),
}
WEBORGANIZER_CONFIDENT_SHARE = {"topic": 0.86, "format": 0.79}
EXTRA_LABELS = ("Other", "Bad example")
CONFIDENCE = 0.75
BOOTSTRAP = 1000
MIN_SUPPORT_ENGLISH = 20


def confident_label_accuracy(label_type: str, variant: str, *, shared_only: bool = False,
                             language: str | None = None, min_support: int = 1,
                             seed: int = 0) -> dict[str, object]:
    # Label-average and worst-label recall on documents whose Qwen top label has probability > 0.75,
    # as WebOrganizer's train_classifier.py; optionally on the 24 labels shared with WebOrganizer
    # (documents with an extra top label dropped, prediction = argmax over the shared labels).
    names = LABELS[label_type]
    targets, probs = TARGETS[label_type], PROBS[label_type, variant]
    keep_labels = np.array([name not in EXTRA_LABELS for name in names]) if shared_only else np.ones(len(names), bool)
    target_top = targets.argmax(axis=1)
    predicted = np.where(keep_labels, probs, -np.inf).argmax(axis=1)
    mask = targets.max(axis=1) > CONFIDENCE
    share_confident = mask.mean() if language is None else mask[DOCS[label_type]["language"].to_numpy() == language].mean()
    mask &= keep_labels[target_top]
    if language is not None:
        mask &= DOCS[label_type]["language"].to_numpy() == language
    target_top, correct = target_top[mask], (predicted == targets.argmax(axis=1))[mask]

    def recalls(index: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        support = np.bincount(target_top[index], minlength=len(names))
        hits = np.bincount(target_top[index], weights=correct[index], minlength=len(names))
        return support, hits

    support, hits = recalls(np.arange(len(target_top)))
    used = keep_labels & (support >= min_support)
    recall = hits[used] / support[used]
    rng = np.random.default_rng(seed)
    averages, worsts = [], []
    for _ in range(BOOTSTRAP):
        sample_support, sample_hits = recalls(rng.integers(0, len(target_top), len(target_top)))
        present = used & (sample_support > 0)
        sample_recall = sample_hits[present] / sample_support[present]
        averages.append(sample_recall.mean())
        worsts.append(sample_recall.min())
    worst_index = np.flatnonzero(used)[recall.argmin()]
    return {
        "Confident share": share_confident,
        "Documents": int(mask.sum()),
        "Labels": int(used.sum()),
        "Accuracy": correct.mean(),
        "Label average": recall.mean(),
        "Average low": np.percentile(averages, 2.5), "Average high": np.percentile(averages, 97.5),
        "Worst label": recall.min(),
        "Worst low": np.percentile(worsts, 2.5), "Worst high": np.percentile(worsts, 97.5),
        "Worst label name": f"{names[worst_index]} ({int(support[worst_index]):,})",
    }


rows = {}
for label_type in LABEL_TYPES:
    for variant in VARIANTS:
        average, worst = WEBORGANIZER[label_type, variant]
        rows[label_type, VARIANT_NAMES[variant], "WebOrganizer (English, 24 labels)"] = {
            "Confident share": WEBORGANIZER_CONFIDENT_SHARE[label_type], "Label average": average,
            "Worst label": worst, "Labels": 24}
        rows[label_type, VARIANT_NAMES[variant], "Ours, 26 labels, 128 languages"] = confident_label_accuracy(
            label_type, variant)
        rows[label_type, VARIANT_NAMES[variant], "Ours, 24 shared labels, 128 languages"] = confident_label_accuracy(
            label_type, variant, shared_only=True)
        english = confident_label_accuracy(label_type, variant, shared_only=True, language="english",
                                           min_support=MIN_SUPPORT_ENGLISH)
        english.update({"Worst label": np.nan, "Worst low": np.nan, "Worst high": np.nan, "Worst label name": ""})
        rows[label_type, VARIANT_NAMES[variant], "Ours, 24 shared labels, English only"] = english
COMPARISON = pd.DataFrame.from_dict(rows, orient="index")
COMPARISON.index.names = ["Label type", "Input", "Model"]

# The 26-label values must equal finetune.py's own conf75 metrics in pred_results.json
# (which covers all documents, so only without EXCLUDE_LEAKED).
for label_type in (LABEL_TYPES if not EXCLUDE_LEAKED else ()):
    for variant in VARIANTS:
        saved = json.loads((prediction_dir(label_type, variant) / "pred_results.json").read_text(encoding="utf-8"))
        mine = COMPARISON.loc[(label_type, VARIANT_NAMES[variant], "Ours, 26 labels, 128 languages")]
        for key, column in (("test_accuracy_label_average_conf75", "Label average"),
                            ("test_accuracy_label_min_conf75", "Worst label")):
            if abs(saved[key] - mine[column]) > 1e-6:
                raise ValueError(f"{label_type}/{variant}: {column} {mine[column]} here, {key} {saved[key]} saved")
print("The 26-label values match accuracy_label_average_conf75 and accuracy_label_min_conf75 in pred_results.json"
      if not EXCLUDE_LEAKED else "Duplicated documents are excluded, so the check against pred_results.json is skipped")


def interval(low: float, high: float) -> str:
    return "" if pd.isna(low) else f"{low:.1%}–{high:.1%}"


shown = pd.DataFrame({
    "Confident share": COMPARISON["Confident share"],
    "Labels": COMPARISON["Labels"],
    "Accuracy": COMPARISON["Accuracy"],
    "Label average": COMPARISON["Label average"],
    "95% interval": [interval(*pair) for pair in zip(COMPARISON["Average low"], COMPARISON["Average high"])],
    "Worst label": COMPARISON["Worst label"],
    "95% interval ": [interval(*pair) for pair in zip(COMPARISON["Worst low"], COMPARISON["Worst high"])],
    "Worst label (confident documents)": COMPARISON["Worst label name"].fillna(""),
})
display(shown.style.format({"Confident share": "{:.0%}", "Accuracy": "{:.1%}", "Label average": "{:.1%}", "Worst label": "{:.1%}",
                            "Labels": "{:.0f}"}, na_rep="—"))

The same comparison as a figure, for all 128 languages: label-average accuracy (filled) and
worst-label accuracy (hollow) on the confident documents, with WebOrganizer's values as
vertical reference lines (solid: label average, dashed: worst label) in each input's colour.

In [ ]:
# English only is left out: its label average covers only the few labels with enough English documents.
MODELS = ["Ours, 26 labels, 128 languages", "Ours, 24 shared labels, 128 languages"]
fig, axes = plt.subplots(1, 2, figsize=(13, 3.6), sharey=True, layout="constrained")
for ax, label_type in zip(axes, LABEL_TYPES):
    for offset, variant in zip((0.14, -0.14), VARIANTS):
        name = VARIANT_NAMES[variant]
        average, worst = WEBORGANIZER[label_type, variant]
        ax.axvline(average * 100, color=COLORS[variant], lw=1.2, alpha=0.8)
        ax.axvline(worst * 100, color=COLORS[variant], lw=1.2, ls="--", alpha=0.8)
        for row, model in enumerate(MODELS):
            values = COMPARISON.loc[(label_type, name, model)]
            y = row + offset
            ax.errorbar(values["Label average"] * 100, y,
                        xerr=[[100 * (values["Label average"] - values["Average low"])],
                              [100 * (values["Average high"] - values["Label average"])]],
                        fmt="o", ms=8, color=COLORS[variant], elinewidth=2, capsize=0,
                        label=f"{name}: label average" if row == 0 else None)
            if not pd.isna(values["Worst label"]):
                ax.errorbar(values["Worst label"] * 100, y,
                            xerr=[[100 * (values["Worst label"] - values["Worst low"])],
                                  [100 * (values["Worst high"] - values["Worst label"])]],
                            fmt="o", ms=8, mfc="white", mec=COLORS[variant], mew=1.8, color=COLORS[variant],
                            elinewidth=1.2, capsize=0, label=f"{name}: worst label" if row == 0 else None)
    ax.set_yticks(range(len(MODELS)), [model.replace("Ours, ", "").replace(", ", "\n", 1) for model in MODELS])
    ax.invert_yaxis()
    ax.grid(axis="y", visible=False)
    ax.set_xlabel("Accuracy on documents with labeller confidence > 75% (%)")
    ax.set_title(f"{label_type.capitalize()} (lines: WebOrganizer)")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="outside lower center", ncol=4, fontsize=9)
fig.suptitle(f"Confident-subset accuracy ({WHEN}) compared with WebOrganizer's Table 7", fontweight="bold")
plt.show()

In [ ]:
# Per-label recall on the confident documents (26 labels, all languages), lowest first:
# which labels pull the worst-label number down.
for label_type in LABEL_TYPES:
    names, targets = LABELS[label_type], TARGETS[label_type]
    confident = targets.max(axis=1) > CONFIDENCE
    table = {}
    for variant in VARIANTS:
        frame = FRAMES[label_type, variant][confident]
        table[f"Recall, {VARIANT_NAMES[variant]}"] = frame.groupby("target")["correct"].mean()
    table = pd.DataFrame(table)
    table.insert(0, "Confident documents", FRAMES[label_type, "url"][confident]["target"].value_counts())
    table.index = [names[index] for index in table.index]
    worst = WEBORGANIZER[label_type, "url"][1]
    below = (table[f"Recall, {VARIANT_NAMES['url']}"] < worst).sum()
    display(Markdown(f"**{label_type.capitalize()}:** {below} of {len(table)} labels are below WebOrganizer's worst "
                     f"label ({worst:.1%}) with the URL; the eight lowest:"))
    display(table.sort_values(f"Recall, {VARIANT_NAMES['url']}").head(8).style.format(
        {"Confident documents": "{:,}", f"Recall, {VARIANT_NAMES['url']}": "{:.1%}",
         f"Recall, {VARIANT_NAMES['nourl']}": "{:.1%}"}))

**What the comparison shows:** *to be written from the final results.* At half an epoch, on
the validation split, format was already above WebOrganizer's reported numbers on the 24
shared labels (label average 94.4% vs 91.8%, worst label 84.5% vs 80.5%, with the URL) and
topic slightly below (92.1% vs 93.5%, worst label 83.7% vs 87.1%); see
`classifier_half_epoch.ipynb`.

**Why this is not a like-for-like comparison:**

- **Different labellers define different subsets.** Each project scores the documents its
  own labeller was confident about. Llama-3.1-405B was confident on 86% of WebOrganizer's
  topic and 79% of its format annotations; Qwen3.8-27B's shares are in the *Confident share*
  column above (on the validation split, 83% and 70%). A more selective subset holds clearer
  cases, which favours our format numbers.
- **Harder data:** 128 languages, including noisy and poor-quality HPLT pages (section 4 has
  the results by source), against English pages only.
- **Training length:** one epoch over 6.85M documents, against 5 epochs in each of
  WebOrganizer's two stages (1M and 100K documents).
- **The 24-label scores are a simulation:** the models were trained with 26 labels and
  restricted to 24 afterwards, which is close to, but not the same as, a 24-label model.
- **Test, not validation:** WebOrganizer reports its validation set; these are test documents,
  unseen also in checkpoint selection, and without duplicates of training documents.

## 11. Duplicates of training documents

Two ways the training split can leak into the evaluation (see the duplicate check in
section 2):

* **Duplicated text:** a document whose text is a near or exact duplicate of a training
  document. The model has in effect seen it with its label, so it is easier. The first
  table compares the results with and without these documents (which is what
  `EXCLUDE_LEAKED` switches).
* **Repeated URL:** a document whose URL also occurs in the training split, usually with
  different text (the same address crawled at another time, or a page that changes). Only
  the URL models see the URL, so if they had learned URL → label, they would gain more on
  these documents than elsewhere, and where Qwen labelled the evaluated page differently
  from its training copy, they would tend to repeat the training copy's label.

In [ ]:
def scores(frame: pd.DataFrame) -> dict[str, float]:
    confident = frame[frame["target_confidence"] > CONFIDENCE]
    recall = confident.groupby("target")["correct"].mean()
    return {
        "Documents": len(frame),
        "Accuracy": frame["correct"].mean(),
        "Label-average accuracy": frame.groupby("target")["correct"].mean().mean(),
        "JSD (bits)": frame["js"].mean(),
        "Confident: label average": recall.mean(),
        "Confident: worst label": recall.min(),
    }


rows = {}
for label_type in LABEL_TYPES:
    for variant in VARIANTS:
        frame = FRAMES_ALL[label_type, variant]
        rows[label_type, VARIANT_NAMES[variant], "All documents"] = scores(frame)
        rows[label_type, VARIANT_NAMES[variant], "Without duplicated documents"] = scores(frame[~LEAKED[label_type]])
leak_table = pd.DataFrame.from_dict(rows, orient="index")
leak_table.index.names = ["Label type", "Input", "Documents"]
display(leak_table.style.format({"Documents": "{:,.0f}", "Accuracy": "{:.2%}", "Label-average accuracy": "{:.2%}",
                                 "JSD (bits)": "{:.4f}", "Confident: label average": "{:.2%}",
                                 "Confident: worst label": "{:.2%}"}))

for label_type in LABEL_TYPES:
    for variant in VARIANTS:
        frame = FRAMES_ALL[label_type, variant]
        print(f"{label_type}, {VARIANT_NAMES[variant]}: accuracy on duplicated documents "
              f"{frame.loc[LEAKED[label_type], 'correct'].mean():.1%}, on the others "
              f"{frame.loc[~LEAKED[label_type], 'correct'].mean():.1%}")

The languages with the most duplicated documents, and their accuracy (URL models) with and
without them:

In [ ]:
languages = DOCS_ALL["topic"]["language"]
leaked_per_language = pd.Series(LEAKED["topic"]).groupby(languages).sum().sort_values(ascending=False)
rows = []
for language in leaked_per_language.index[:12]:
    mine = (languages == language).to_numpy()
    row = {"Language": LANGUAGE_NAMES[language], "Documents": int(mine.sum()),
           "Duplicated": int((mine & LEAKED["topic"]).sum())}
    for label_type in LABEL_TYPES:
        correct = FRAMES_ALL[label_type, "url"]["correct"].to_numpy()
        row[f"{label_type.capitalize()}, all"] = correct[mine].mean()
        row[f"{label_type.capitalize()}, without"] = correct[mine & ~LEAKED[label_type]].mean()
    rows.append(row)
by_language = pd.DataFrame(rows)
display(by_language.style.format({column: "{:.1%}" for column in by_language.columns if "," in column}
                                 | {"Documents": "{:,}", "Duplicated": "{:,}"}).hide(axis="index"))

### Repeated URLs: do the URL models memorise them?

The evaluated documents in four groups: no repeat in training; the same URL as a training
document with similar text (estimated Jaccard ≥ 0.8); the same URL with different text;
similar text under another URL. *URL gain* is the URL model's accuracy minus the text-only
model's on the same documents (paired 95% interval). Memorisation would show as a larger
gain in the "same URL, different text" group than in the "no repeat" group.

In [ ]:
def paired_gain(a: np.ndarray, b: np.ndarray) -> tuple[float, float]:
    difference = a.astype(float) - b.astype(float)
    if len(difference) < 2:
        return float("nan"), float("nan")
    return difference.mean(), 1.96 * difference.std(ddof=1) / math.sqrt(len(difference))


for label_type in LABEL_TYPES:
    count = len(DOCS_ALL[label_type])
    repeats = URL_REPEATS[label_type]
    similarity = np.full(count, np.nan)
    twin = np.full(count, -1)
    similarity[repeats["position"]] = repeats["similarity"].fillna(0.0)
    twin[repeats["position"]] = repeats["twin_label"]
    url_repeat, text_repeat = twin >= 0, LEAKED[label_type]
    groups = {
        "No repeat in training": ~url_repeat & ~text_repeat,
        "Same URL, similar text": url_repeat & (similarity >= 0.8),
        "Same URL, different text": url_repeat & (similarity < 0.8),
        "Similar text, other URL": text_repeat & ~url_repeat,
    }
    url_correct = FRAMES_ALL[label_type, "url"]["correct"].to_numpy()
    text_correct = FRAMES_ALL[label_type, "nourl"]["correct"].to_numpy()
    rows, gains = [], {}
    for name, mask in groups.items():
        gain, half = paired_gain(url_correct[mask], text_correct[mask])
        gains[name] = (gain, half)
        rows.append({"Group": name, "Documents": int(mask.sum()), "URL + text": url_correct[mask].mean(),
                     "Text only": text_correct[mask].mean(), "URL gain (points)": 100 * gain,
                     "± (95%)": 100 * half})
    display(Markdown(f"**{label_type.capitalize()}**"))
    display(pd.DataFrame(rows).style.format({"Documents": "{:,}", "URL + text": "{:.1%}", "Text only": "{:.1%}",
                                             "URL gain (points)": "{:+.1f}", "± (95%)": "{:.1f}"},
                                            na_rep="—").hide(axis="index"))
    (gain_c, half_c), (gain_a, half_a) = gains["Same URL, different text"], gains["No repeat in training"]
    different = groups["Same URL, different text"]
    target = FRAMES_ALL[label_type, "url"]["target"].to_numpy()
    relabelled = different & (target != twin)
    copies = {variant: (FRAMES_ALL[label_type, variant]["predicted"].to_numpy() == twin)[relabelled]
              for variant in VARIANTS}
    copy_gain, copy_half = paired_gain(copies["url"], copies["nourl"])
    display(Markdown(
        f"URL gain with the same URL and different text minus the gain with no repeat: "
        f"{100 * (gain_c - gain_a):+.1f} ± {100 * math.sqrt(half_c ** 2 + half_a ** 2):.1f} points. "
        f"Qwen gave {relabelled.sum():,} of those {different.sum():,} documents a different top label from their "
        f"training copy; there the URL model predicts the training copy's label for {copies['url'].mean():.1%} and "
        f"the text-only model for {copies['nourl'].mean():.1%} (difference {100 * copy_gain:+.1f} ± "
        f"{100 * copy_half:.1f} points)."))

## 12. Summary

*Update once the final models have been trained and evaluated on the test split.*